# Importation

In [ ]:
import os
import math
import copy
import json
import torch
import warnings
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
#from torch.utils.data.sampler import WeightedRandomSampler
from typing import Dict, List, Tuple, Any
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils import weight_norm
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import tensorflow.keras as tf_keras
import tensorflow as tf
from transformers import AutoTokenizer
from transformers import DistilBertModel, DistilBertTokenizerFast


from models import SA_LSTM_Classification_Model, SelfAttention, BERTLSTMClassifier

# Chargement modèles

In [ ]:
MODEL_VIDEO_PATH = "./video/best_sa_lstm_53,1.pt"
MODEL_AUDIO_PATH = "./audio/yamnet_classifier.pt"
MODEL_TEXT_PATH = "./texte/best_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

video_model = torch.load(MODEL_VIDEO_PATH, map_location=DEVICE, weights_only=False)
audio_model = torch.load(MODEL_AUDIO_PATH, map_location=DEVICE, weights_only=False)
state_dict = torch.load(MODEL_TEXT_PATH, map_location="cpu", weights_only=True)
tokenizer = DistilBertTokenizerFast.from_pretrained("./texte/finetuned_bert")


text_model = BERTLSTMClassifier(
    output_dim=20, hidden_dim=256, n_layers=2, dropout=0.4, bert_dir="./texte/finetuned_bert"
).to(DEVICE)

state = torch.load(MODEL_TEXT_PATH, map_location=DEVICE)
missing, unexpected = text_model.load_state_dict(state, strict=False)
if missing or unexpected:
    print("⚠️ State dict diff:", {"missing": missing, "unexpected": unexpected})


RuntimeError: Error(s) in loading state_dict for LSTMClassifier:
	size mismatch for fc2.weight: copying a param with shape torch.Size([384, 512]) from checkpoint, the shape in current model is torch.Size([560, 512]).
	size mismatch for fc2.bias: copying a param with shape torch.Size([384]) from checkpoint, the shape in current model is torch.Size([560]).
	size mismatch for fc3.weight: copying a param with shape torch.Size([256, 384]) from checkpoint, the shape in current model is torch.Size([512, 560]).
	size mismatch for fc3.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for fc4.weight: copying a param with shape torch.Size([20, 256]) from checkpoint, the shape in current model is torch.Size([20, 512]).

In [21]:
JSON_PATH = "./infos/train_val_videodatainfo.json"
VIDEO_FEAT_DIR = "./video/features_finetuned"
AUDIO_FEAT_DIR = "./audio/audios_embeddings"

# Chargements features

In [23]:

# === Chemins par défaut ===
JSON_PATH = globals().get("JSON_PATH", "train_val_videodatainfo.json")
VIDEO_FEAT_DIR = globals().get("VIDEO_FEAT_DIR", "features_video")
AUDIO_FEAT_DIR = globals().get("AUDIO_FEAT_DIR", "features_audio")
TEXT_EMB_TYPE = globals().get("TEXT_EMB_TYPE", "captions")  # pour clarté

class MultiModalDataset(Dataset):
    """
    Dataset multimodal pour MSR-VTT.
    Retourne (feat_video, feat_audio, list_captions, label)
    """
    def __init__(self, split: str, json_path: str = JSON_PATH,
                 video_dir: str = VIDEO_FEAT_DIR, audio_dir: str = AUDIO_FEAT_DIR):
        super().__init__()
        self.split = split
        self.json_path = json_path
        self.video_dir = video_dir
        self.audio_dir = audio_dir

        self.samples: List[Dict[str, Any]] = []
        self.class_counts: Dict[int, int] = {}
        self._index_from_json()

    def _index_from_json(self):
        if not os.path.isfile(self.json_path):
            raise FileNotFoundError(f"JSON file not found: {self.json_path}")
        with open(self.json_path, "r") as f:
            data = json.load(f)

        videos = data.get("videos", [])
        for v in videos:
            if v.get("split") != self.split:
                continue

            vid = v.get("video_id")
            label = int(v.get("category"))
            captions = v.get("captions", [])  # liste de captions texte
            video_path = os.path.join(self.video_dir, f"{vid}.npy")
            audio_path = os.path.join(self.audio_dir, f"{vid}.npy")

            if not os.path.exists(video_path):
                warnings.warn(f"[{self.split}] Missing video features: {video_path}")
                continue

            self.samples.append({
                "video": video_path,
                "audio": audio_path if os.path.exists(audio_path) else None,
                "captions": captions,
                "label": label,
            })
            self.class_counts[label] = self.class_counts.get(label, 0) + 1

        if not self.samples:
            raise RuntimeError(f"No multimodal samples found for split='{self.split}'")

        # Mise à jour du nombre de classes global
        unique_classes = sorted(set(s["label"] for s in self.samples))
        if "NUM_CLASSES" in globals():
            globals()["NUM_CLASSES"] = max(unique_classes) + 1

        print(f"[{self.split}] Indexed {len(self.samples)} samples across {len(unique_classes)} classes.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        s = self.samples[idx]
        label = s["label"]

        # === Charger les features vidéo ===
        video_feats = np.load(s["video"], allow_pickle=True)
        video_feats = torch.tensor(video_feats, dtype=torch.float32)

        # === Charger les features audio (ou zéro) ===
        if s["audio"] and os.path.exists(s["audio"]):
            audio_feats = np.load(s["audio"], allow_pickle=True)
            audio_feats = torch.tensor(audio_feats, dtype=torch.float32)
        else:
            # Pas de son → vecteur nul (même dim que vidéo ou 128 par défaut)
            audio_feats = torch.zeros_like(video_feats) if video_feats.ndim == 2 else torch.zeros((1, video_feats.shape[-1]))

        # === Convertir les captions ===
        captions = s["captions"]
        if isinstance(captions, list):
            # Embedding via ton tokenizer ou modèle textuel plus tard dans le pipeline
            captions_emb = captions  # on laisse brut ici
        else:
            captions_emb = [str(captions)]

        return video_feats, audio_feats, captions_emb, int(label)




def collate_fn_multimodal(batch):
    """
    Prépare un batch multimodal :
      - pad vidéo/audio
      - tokenize captions avec DistilBERT
      - assemble dans un dict pour un modèle multimodal
    """
    videos, audios, captions, labels = zip(*batch)

    # --- Pad vidéo ---
    video_lens = [v.shape[0] if v.ndim >= 2 else 1 for v in videos]
    max_vlen = max(video_lens)
    vdim = videos[0].shape[-1]
    padded_videos = torch.zeros((len(videos), max_vlen, vdim), dtype=torch.float32)
    for i, v in enumerate(videos):
        L = v.shape[0]
        padded_videos[i, :L] = v[:L]

    # --- Pad audio ---
    audio_lens = [a.shape[0] if a.ndim >= 2 else 1 for a in audios]
    max_alen = max(audio_lens)
    adim = audios[0].shape[-1]
    padded_audios = torch.zeros((len(audios), max_alen, adim), dtype=torch.float32)
    for i, a in enumerate(audios):
        L = a.shape[0]
        padded_audios[i, :L] = a[:L]

    # --- Tokenize captions ---
    # Chaque sample peut avoir plusieurs captions → on en prend une aléatoire pour ce batch
    texts = [np.random.choice(caps) if isinstance(caps, list) and len(caps) > 0 else "" for caps in captions]
    text_enc = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")

    labels = torch.tensor(labels, dtype=torch.long)

    batch_out = {
        "video": padded_videos,
        "audio": padded_audios,
        "text_input_ids": text_enc["input_ids"],
        "text_attention_mask": text_enc["attention_mask"],
        "labels": labels,
    }

    return batch_out


def setup_multimodal_pipeline(
    batch_size=None,
    shuffle=False,
    num_workers=0,
    use_weighted_sampler=False,
    split=None,
):
    bs = batch_size or globals().get("BATCH_SIZE", 32)
    if split is None:
        split = "train" if shuffle else "validate"

    dataset = MultiModalDataset(split=split)

    labels = [s["label"] for s in dataset.samples]
    num_classes = max(labels) + 1
    class_counts = {c: labels.count(c) for c in set(labels)}
    class_weights = torch.ones(num_classes, dtype=torch.float32)
    for c, cnt in class_counts.items():
        class_weights[c] = 1.0 / max(cnt, 1)

    sampler = None
    if use_weighted_sampler:
        sample_weights = [class_weights[lbl].item() for lbl in labels]
        sampler = WeightedRandomSampler(torch.tensor(sample_weights), len(sample_weights))
        shuffle = False

    loader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn_multimodal,
    )
    return loader, num_classes


# Construction Cross-modal audio+vidéo+texte

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiModalUnifier(nn.Module):
    def __init__(self, video_model, audio_model, text_model, num_classes, hidden_dim=256, dropout=0.3):
        """
        video_model, audio_model, text_model : sous-modèles pré-entraînés
        num_classes : nombre total de classes
        hidden_dim : dimension du MLP
        """
        super().__init__()
        self.video_model = video_model
        self.audio_model = audio_model
        self.text_model = text_model
        self.num_classes = num_classes

        print(text_model)

        # Geler les sous-modèles si nécessaire (par défaut, on les met en eval)
        for m in [self.video_model, self.audio_model, self.text_model]:
            for p in m.parameters():
                p.requires_grad = False
            m.eval()

        # Taille d’entrée du MLP = somme des dimensions de logits des 3 modèles
        self.input_dim = num_classes * 3

        # Petit MLP de fusion
        self.mlp = nn.Sequential(
            nn.Linear(self.input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, batch):
        """
        batch: dict venant du collate_fn_multimodal
        Retourne les logits fusionnés.
        """

        # --- 1️⃣ Vidéo ---
        video_feats = batch["video"]
        video_feats = video_feats.view(1, 4, 1408)
        video_logits = self.video_model(video_feats)  # [B, C]

        # --- 2️⃣ Audio ---
        audio_feats = batch["audio"]
        audio_logits = self.audio_model(audio_feats)  # [B, C]

        # --- 3️⃣ Texte ---
        input_ids = batch["text_input_ids"]
        attention_mask = batch["text_attention_mask"]

        # Certaines vidéos ont plusieurs captions → on peut soit batcher, soit moyenner
        if input_ids.ndim == 3:
            # [B, N_captions, L]
            B, N, L = input_ids.shape
            text_logits_all = []
            for i in range(N):
                logits_i = self.text_model(input_ids[:, i, :], attention_mask[:, i, :])  # [B, C]
                text_logits_all.append(logits_i)
            text_logits = torch.stack(text_logits_all, dim=1).mean(dim=1)  # [B, C]
        else:
            text_logits = self.text_model(input_ids, attention_mask)  # [B, C]

        # --- 4️⃣ Fusion tardive ---
        fused = torch.cat([video_logits, audio_logits, text_logits], dim=1)  # [B, 3*C]
        output = self.mlp(fused)

        return output


# Entrainement

In [17]:
# --- Constants ---

# Will be updated by setup_data_pipeline based on dataset folders
NUM_CLASSES = 20



BATCH_SIZE = 128          # Keep or adjust based on GPU memory
NUM_EPOCHS = 100          # Reduced max epochs (early stopping will likely trigger sooner)
LEARNING_RATE = 1e-4      # Slightly reduced learning rate
WEIGHT_DECAY = 1e-3       # Significantly increased weight decay for regularization
MIXUP_ALPHA = 0.1         # Slightly increased MixUp strength
LABEL_SMOOTHING = 0.05     # Slightly increased label smoothing
EARLY_STOP_PATIENCE = 15  # Significantly reduced early stopping patience
SCHEDULER_PATIENCE = 5    # Patience for LR reduction
SCHEDULER_FACTOR = 0.5    # Factor to reduce LR by
MIN_LR = 1e-6             # Minimum learning rate

CLASSIFIER_DROPOUT = 0.3


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
from tqdm import tqdm

# --- MixUp (sur les logits, version soft) ---
def logits_mixup(logits, labels, alpha=0.2):
    """MixUp applied to logits directly."""
    if alpha <= 0:
        return logits, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = logits.size(0)
    if batch_size == 1:
        return logits, labels, labels, 1.0
    index = torch.randperm(batch_size, device=logits.device)
    mixed_logits = lam * logits + (1 - lam) * logits[index, :]
    y_a, y_b = labels, labels[index]
    return mixed_logits, y_a, y_b, lam


# --- Train one epoch ---
def train_one_epoch_unifier(model, dataloader, optimizer, criterion, device, mixup_alpha=0.2, grad_clip_value=5.0):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        # Préparation des données
        for k, v in batch.items():
            if torch.is_tensor(v):
                batch[k] = v.to(device)

        labels = batch["labels"]

        optimizer.zero_grad()
        logits = model(batch)  # le modèle unifie déjà vidéo/audio/texte

        # --- MixUp sur logits ---
        mixed_logits, y_a, y_b, lam = logits_mixup(logits, labels, alpha=mixup_alpha)
        loss = lam * criterion(mixed_logits, y_a) + (1 - lam) * criterion(mixed_logits, y_b)

        loss.backward()

        if grad_clip_value > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_value)
        optimizer.step()

        # --- Statistiques ---
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, 100 * total_correct / total_samples


# --- Validation ---
def evaluate_unifier(model, dataloader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            for k, v in batch.items():
                if torch.is_tensor(v):
                    batch[k] = v.to(device)
            labels = batch["labels"]
            logits = model(batch)
            loss = criterion(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, 100 * total_correct / total_samples


# --- Full Pipeline ---
def run_pipeline_unifier(train_loader, val_loader, num_classes, save_path="best_unifier.pt"):
    print("\n--- Running Unifier Training Pipeline ---")
    print(f"Device: {DEVICE}")
    print(f"  LR: {LEARNING_RATE}, Weight Decay: {WEIGHT_DECAY}")
    print(f"  MixUp α: {MIXUP_ALPHA}, Dropout: {CLASSIFIER_DROPOUT}")
    print("-" * 40)

    # --- Model ---
    unifier = MultiModalUnifier(
        video_model=video_model,
        audio_model=audio_model,
        text_model=text_model,
        num_classes=num_classes,
        hidden_dim=HIDDEN_DIM,
        dropout=CLASSIFIER_DROPOUT
    ).to(DEVICE)

    # --- Loss / Optimizer / Scheduler ---
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(unifier.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=SCHEDULER_FACTOR,
                                               patience=SCHEDULER_PATIENCE, min_lr=MIN_LR)

    best_val_acc, no_improve_epochs = 0.0, 0

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        train_loss, train_acc = train_one_epoch_unifier(unifier, train_loader, optimizer, criterion, DEVICE, MIXUP_ALPHA)
        val_loss, val_acc = evaluate_unifier(unifier, val_loader, criterion, DEVICE)

        scheduler.step(val_acc)
        lr = optimizer.param_groups[0]['lr']

        print(f"Train Acc: {train_acc:.2f}% (Loss: {train_loss:.4f}) | "
              f"Val Acc: {val_acc:.2f}% (Loss: {val_loss:.4f}) | LR: {lr:.6f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(unifier.state_dict(), save_path)
            print(f"   ✅ New best val acc: {val_acc:.2f}% → Saved to {save_path}")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= EARLY_STOP_PATIENCE:
                print(f"--- Early stopping after {epoch+1} epochs (no improvement for {EARLY_STOP_PATIENCE}). ---")
                break

    print(f"\n--- Training Complete. Best Val Acc: {best_val_acc:.2f}% ---")
    return unifier


In [24]:
train_loader = setup_multimodal_pipeline(
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=os.cpu_count() // 2,
    use_weighted_sampler=True,
    split="train",
)

val_loader = setup_multimodal_pipeline(
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=os.cpu_count() // 2,
    use_weighted_sampler=False,
    split="validate",
)

[train] Indexed 6513 samples across 20 classes.
[validate] Indexed 497 samples across 20 classes.


In [32]:
run_pipeline_unifier(train_loader, val_loader, NUM_CLASSES, save_path="best_unifier.pt")


--- Running Unifier Training Pipeline ---
Device: cuda
  LR: 0.0001, Weight Decay: 0.001
  MixUp α: 0.1, Dropout: 0.3
----------------------------------------
OrderedDict([('embedding.weight', tensor([[ 4.5099e-01, -6.7025e-01, -8.2410e-01,  ...,  3.0045e-01,
         -8.5570e-03,  1.7786e-01],
        [-2.4673e-41,  2.4540e-41, -2.4656e-41,  ..., -2.4664e-41,
         -2.4702e-41,  2.4590e-41],
        [-2.4607e-41, -2.4575e-41,  2.4755e-41,  ...,  2.4667e-41,
         -1.6871e-02,  2.4799e-41],
        ...,
        [-2.4631e-41, -2.4747e-41,  2.4656e-41,  ..., -2.4758e-41,
         -2.4971e-41, -2.4612e-41],
        [-2.4538e-41, -2.4712e-41,  2.4583e-41,  ...,  2.4678e-41,
          2.4535e-41,  2.4636e-41],
        [ 2.4699e-41, -2.4541e-41,  2.4555e-41,  ..., -2.4619e-41,
         -2.4542e-41, -2.4611e-41]], device='cuda:0')), ('lstm.weight_ih_l0', tensor([[ 0.0082, -0.0111,  0.0075,  ...,  0.0177,  0.0263, -0.0024],
        [-0.0055,  0.0161,  0.0266,  ..., -0.0834, -0.0374, -0.

AttributeError: 'collections.OrderedDict' object has no attribute 'parameters'

# Sauvegarde

# Evaluation